In [ ]:
import numpy as np

In [ ]:
words = open('names.txt', 'r').read().splitlines()
words[:10]

In [ ]:
len(words)

In [ ]:
min(len(w) for w in words)

In [ ]:
max(len(w) for w in words)

# bigram language model

* In a bigram model, we always work with two characters at a time.
* We only look at one character and try to predict the next character in the sequence.
* Although we may have a lot of information, we are always just looking at the previous character to predict the next one, making this a very simple and weak language model."

In [ ]:
for w in words[:1]:
    for ch1, ch2 in zip (w, w[1:]):
        print(ch1,ch2)

In [ ]:
w

In [ ]:
w[1:]

In [ ]:
for w in words[:3]:
    chs = ['<S>'] + list(w) + ['<E>'] # so in this way we can detect the most of start and end char
    for ch1, ch2 in zip (chs, chs[1:]):
        print(ch1,ch2)

* we are gonna count how often any one of these combinations occurs in the training set

In [ ]:
b = {}
for w in words[:3]:
    chs = ['<S>'] + list(w) + ['<E>'] # so in this way we can detect the mot of start and end char
    for ch1, ch2 in zip (chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1
        print(ch1,ch2)

In [ ]:
b

* In this point we can figure out a has a potential being last char. (('a', '< E >'): 3,)
* now lets check on all words

In [ ]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>'] # so in this way we can detect the mot of start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1


In [ ]:
#b

In [ ]:
sorted(b.items(), key = lambda kv: -kv[1]) # lets check common ones, -kv[1]-> 1 means count number

* we're gonna store this information in 3d array
* rows = first chc, second = second chc

In [ ]:
import torch

In [ ]:
a = torch.zeros((3,5), dtype = torch.int32) # 3x5 array
a

In [ ]:
a.dtype

* Tensors allow us to manipulate individual entries efficiently.

In [ ]:
a[1,3] = 2

In [ ]:
a

In [ ]:
N = torch.zeros((28,28), dtype = torch.int32) # 26 letter and 2 mark sign


In [ ]:

for w in words:
    chs = ['<S>'] + list(w) + ['<E>'] # so in this way we can detect the most of repeat start and end char
    for ch1, ch2 in zip (chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

* now we need the lookup table from char to int

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i for i,s in enumerate(chars)} # string to integer (stoi)
stoi['<S>'] = 26 # we need to add two mark manually 
stoi['<E>'] = 27
stoi

In [ ]:
for w in words:
    chs = ['<S>'] + list(w) + ['<E>'] 
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1,ix2] +=1

N


* Lets try to visualize nicer

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(N)

* Still looking ugly...
* First off all we are going to need to invert stoi to reverse dictionary (stoi -> itos)

In [ ]:
itos = {i:s for s,i in stoi.items()}
itos

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

* we have a some problem on unique char in plot lets fix it

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [ ]:
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [ ]:
N[0]

In [ ]:
p = N[0].float() # probabilty
p = p/p.sum()
p

* Now p giving us the probabilty any single char to be first char of a word

In [ ]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator =g)
p = p /p.sum()
p

* https://pytorch.org/docs/stable/generated/torch.multinomial.html

In [ ]:
torch.multinomial(p, num_samples=20, replacement=True,generator =g)
# like this method we are always getting deterministic result

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
print(ix)
itos[ix]

* now we have a first letter lets check it others in loop

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = 0
while True:
    p = N[ix].float()
    p = p / p.sum()
    ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
    print(itos[ix])
    if ix == 0: #thats mean end token
        break

* our word is 'mor.'

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = 0
out = []

while True:

    p = N[ix].float()
    p = p / p.sum()
    ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
    out.append(itos[ix])
    if ix == 0: #thats mean end token
        break
print(''.join(out))

* we are always getting same result because of generator lets change it

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(20):

    ix = 0
    out = []

    while True:

        p = N[ix].float()
        p = p / p.sum()
        ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
        out.append(itos[ix])
        if ix == 0: #thats mean end token
            break
    print(''.join(out))

* Oops. These outputs don't look good (for example, single letters like 'h' as words). The reason is that our model sometimes predicts 'h' as the first letter, but 'h' is not often used as a first character.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(20):

    ix = 0
    out = []

    while True:

        p = N[ix].float()
        p = p / p.sum()
        ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
        out.append(itos[ix])
        if ix == 0: #thats mean end token
            break
    print(''.join(out))

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(20):

    ix = 0
    out = []

    while True:

        # p = N[ix].float()
        # p = p / p.sum()
        p = torch.ones(27) /27
        ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
        out.append(itos[ix])
        if ix == 0: #thats mean end token
            break
    print(''.join(out))

* Now we have a uniform distribution, so everything is equal. Our outputs are coming from an untrained model (where every character has the same probability), and they are garbage.

* first of all we will define new P because the old version of P in our model is not working efficient

In [ ]:
P = N.float()
# P /P.sum() # 

In [ ]:
P.sum() # p.sum is a huge number we have to find new way

In [ ]:
P.sum(0, keepdim=True) 

In [ ]:
P.sum(0, keepdim=True) .shape

* now we have a sum of row vector not sum of all
* but actually we dont wanna [1,27] dimension we want a another way

In [ ]:
P.sum(1, keepdim=True) .shape

In [ ]:
P.sum(1, keepdim=True)

In [ ]:
P = N.float()
P = P /P.sum(1, keepdim= True) 

* this section contains broadcasting sementics read the article (P = P /P.sum(1, keepdim= True))
* https://pytorch.org/docs/stable/notes/broadcasting.html
* !!!CHECKPOINT

In [ ]:
P[0].sum()

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(20):

    ix = 0
    out = []

    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
        out.append(itos[ix])
        if ix == 0: #thats mean end token
            break
    print(''.join(out))

* if we dont use keepdims=True parameter lets check whats gonna  happen.(broadcasting  disable)

In [ ]:
P = N.float()
P/=P.sum(1) 

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(20):

    ix = 0
    out = []

    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
        out.append(itos[ix])
        if ix == 0: #thats mean end token
            break
    print(''.join(out))

* complately meaningless... Thats why broadcasting is important 
* this extra '1' dimension is necessary 
* one more detail we should use P/=P.sum() because otherwise old school expression is creating a new tensor and its not efficency

* Lets evaluate the quality of this model 

In [ ]:
P = N.float()
P = P /P.sum(1, keepdim= True) 

g = torch.Generator().manual_seed(2147483647)
for i in range(5):

    ix = 0
    out = []

    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
        out.append(itos[ix])
        if ix == 0: #thats mean end token
            break
    print(''.join(out))

In [ ]:
for w in words[:3]:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        print(f'{ch1}{ch2}:{prob:.4f}')

* We have 27 possible characters or tokens, and if everything is equally likely, each one has a probability of 0.03 (1/27 = 0.03). So, any probability above 4% means we have learned something useful from these bigrams. If you have a perfect model, these probabilities should be 1 because the model correctly predicts what will come next, especially on the training set.

* how can we summarize these probabilities? -> Likelyhood function
* https://en.wikipedia.org/wiki/Likelihood_function
* Likelihood is the product of these probabilities. When training the model, the likelihood should be as high as possible, indicating a good model.

* for example we have  a 3 probs(a,b,c). 
* * likelihood prob -> a * b* c
* * log likelihood probs -> log(a * b* c) -> log(a) + log(b) + log(c)

In [ ]:
log_likelihood = 0.0
for w in words[:3]:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood+= logprob
        print(f'{ch1}{ch2}:{prob:.4f}{logprob:.4f}')
print( f'{log_likelihood=}')

* How high can the log-likelihood get? If all probabilities are 1, then the log-likelihood will be 0. When all probabilities are lower, the log-likelihood becomes more and more negative. However, we don't want this because, as a loss function, lower values are better since we try to minimize the loss. So, we should invert this function to resemble a loss function. We need a negative log-likelihood function.

In [ ]:
log_likelihood = 0.0
n_l_l = -log_likelihood
for w in words[:3]:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood+= logprob
        print(f'{ch1}{ch2}:{prob:.4f}{logprob:.4f}')
print( f'{log_likelihood=}')
n_l_l = -log_likelihood
print( f'{n_l_l=}')

* negative_log_likelyhood very good for us because lowest it can get is zero and the higher it can get worse off predictions

In [ ]:
log_likelihood = 0.0
n = 0
for w in words[:3]:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood+= logprob
        n+=1
        print(f'{ch1}{ch2}:{prob:.4f}{logprob:.4f}')
print( f'{log_likelihood=}')
n_l_l = -log_likelihood
print( f'{n_l_l=}')
print( f'{n_l_l/n}')

* now our job is find the parameters that minimize the nll
* GOAL: maximize likelihood of the data w.r.t. model parameters (statistical modeling)
* equivalent to maximizing the log likelihood (because log is monotonic)
* equivalent to minimizing the negative log likelihood
* equivalent to minimizing the average negative log likelihood

* log(a*b*c) = log(a) + log(b) + log(c)

In [ ]:
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood+= logprob
        n+=1
        print(f'{ch1}{ch2}:{prob:.4f}{logprob:.4f}')
print( f'{log_likelihood=}')
n_l_l = -log_likelihood
print( f'{n_l_l=}')
print( f'{n_l_l/n}')

* lets check with standart name which is 'mike'

In [ ]:
log_likelihood = 0.0
n = 0
for w in ["mike"]:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood+= logprob
        n+=1
        print(f'{ch1}{ch2}:{prob:.4f}{logprob:.4f}')
print( f'{log_likelihood=}')
n_l_l = -log_likelihood
print( f'{n_l_l=}')
print( f'{n_l_l/n}')

* mike score is (2.249427318572998)
* lets check with absurd name

In [ ]:
log_likelihood = 0.0
n = 0
for w in ["fatijq"]:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood+= logprob
        n+=1
        print(f'{ch1}{ch2}:{prob:.4f}{logprob:.4f}')
print( f'{log_likelihood=}')
n_l_l = -log_likelihood
print( f'{n_l_l=}')
print( f'{n_l_l/n}')

* The result is infinite because the combination 'jq:0.0000-inf' has zero probability. This looks bad, so we should fix this representation even though it's technically correct. We can address this with a simple method called model smoothing. By adding some fake counts to combinations with zero probabilities, we can avoid infinite results.

In [ ]:
P = (N+1).float() # +1 so none of probs of combination can be zero
P = P /P.sum(1, keepdim= True) 

g = torch.Generator().manual_seed(2147483647)
for i in range(5):

    ix = 0
    out = []

    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True,generator =g).item()
        out.append(itos[ix])
        if ix == 0: #thats mean end token
            break
    print(''.join(out))

In [ ]:
log_likelihood = 0.0
n = 0
for w in ["fatijq"]:
    chs = ['.'] + list(w) + ['.'] # so in this way we can detect the most of repeat start and end chac
    for ch1, ch2 in zip (chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood+= logprob
        n+=1
        print(f'{ch1}{ch2}:{prob:.4f}{logprob:.4f}')
print( f'{log_likelihood=}')
n_l_l = -log_likelihood
print( f'{n_l_l=}')
print( f'{n_l_l/n}')

* so there is no more inf result after our fix

* summary
* We have now trained a bigram character language model.
* We checked all counts of bigrams and normalized the rows to get probability distributions.
* We can also use these probabilities to evaluate the model by sampling new words based on these distributions.
* We sampled new words according to those distributions.
* This model approach looks sensible, but now we would like to try a new approach.
* We will end up in a very similar position with the new model, but the approach will look very different as we cast the problem of the bigram character * language model into the neural network framework.

* Our model receives a single character as input, and then a neural network with some weights or parameters (w) outputs the probability distribution for the next character in the sequence. This means our model makes guesses about what is likely to follow the input character. We have a bigram table that provides the true answer, which we can use to calculate the loss value. We will use a gradient descent approach to minimize the loss function.

In [ ]:
# create a training set of bigrams(x,y)
'xs: input char, ys:output char(predict next char)'
xs, ys = [], []

for w in words[:1]: # we get just one word to stay managable to model 
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    print(ch1, ch2)
    xs.append(ix1)
    ys.append(ix2)
    
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs

In [ ]:
ys

* we used torch.tensor but why arent we  using torch.Tensor the reason is "torch.tensor infers the dtype automatically, while torch.Tensor returns a torch.FloatTensor. I would recommend to stick to torch.tensor, which also has arguments like dtype, if you would like to change the type."
* https://stackoverflow.com/questions/51911749/what-is-the-difference-between-torch-tensor-and-torch-tensor

* Now, how are we going to feed these examples into a neural network? It's not as straightforward as just plugging them in because these examples are integers, giving us the index of the character, which we can't use directly. We need to encode the integers using one-hot encoding.
* https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html
* 

In [ ]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=27) # xs ([ 0,  5, 13, 13,  1])
xenc

In [ ]:
xenc.shape

In [ ]:
plt.imshow(xenc)
# you can check ([ 0,  5, 13, 13,  1])

* we always be careful about the data types because we dont wanna put integers value to neural network we want them as  a floating point
* we have a 5 examples

In [ ]:
xenc.dtype


In [ ]:
import torch.nn.functional as F
xenc = F.one_hot(xs, num_classes=27).float() # we can fix like that
xenc

In [ ]:
plt.imshow(xenc)

In [ ]:
xenc.dtype

In [ ]:
xenc.shape

In [ ]:
W = torch.randn((27,1)) # random weight initialize for single neuron
xenc @ W 

* xenc is our input
* @ is a matrix multiplication
* xenc @ W ----> [5, 27] @ [27,1] = [5,1]
* we need a 27 neuron

In [ ]:
W = torch.randn((27,27)) # random weight initialize for 27 neuron
xenc @ W 

* Previously, we had 27 inputs, and now we have 27 neurons in the hidden layer. We don't add an activation function or another layer; this is the simplest neural network, which is a single linear layer.


* In xenc @ 𝑊 we just have some negative and positive numbers, but we want those numbers to somehow represent the probabilities for the next character. Probabilities have a special structure: they are positive numbers and they sum to 1, which doesn't naturally come out of a neural network. Our 27 numbers give us log counts, so we must convert these numbers into the format we need.

In [ ]:
(xenc @ W ).exp()

* Using the exp function, we can convert negative numbers to positive values while keeping positive numbers positive. However, we now have values less than 1, which isn't ideal for our log counts.

In [ ]:
logits = (xenc @ W ) # log -counts
counts = logits.exp() # equivalent N
probs = counts / counts.sum(1, keepdims = True) # normalize counts value
probs

* now every row here to 1
* by the way all this operations are differentiable 
* now time is adjust the w to find best probs 

* SUMMARY
* * 1-we have some inputs (xs) and some labes for the correct next charachter in the sequence (ys) and these are int
* * 2-and generating 27 neuron weights (w) and each neuron receives 27 inputs
* * 3-after that we have to encode all of the inputs with one hot 
* * 4-after then multiple this in the first layer of a neural net to get logits
* * 5-exponentiate the logits to get fake counts sort of 
* * 6-normalize these counts to get probs (!!! step 5-6 called softmax)
* * step 3-4-5-6 called forward pass

* backpropagation time...
* our operation was multiplication addition exponention sum. we know how can backpropagate through them because everything here is a differentiable

In [ ]:
nlls = torch.zeros(5) # inputs coming from 'emma'
for i in range(5):
  # i-th bigram:
  x = xs[i].item() # input character index
  y = ys[i].item() # label character index
  print('--------')
  print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
  print('input to the neural net:', x)
  print('output probabilities from the neural net:', probs[i])
  print('label (actual next character):', y)
  p = probs[i, y]
  print('probability assigned by the net to the the correct character:', p.item())
  logp = torch.log(p)
  print('log likelihood:', logp.item())
  nll = -logp
  print('negative log likelihood:', nll.item())
  nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

* Now we have a loss function and this loss is only made up differentiable operations we can minimize the loss by tuning w.
* so then we can tune minimize the loss and find a good setting of w

* OPTIMIZATION

In [ ]:
xs

In [ ]:
ys

In [ ]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# forward  pass
xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# btw: the last 2 lines here are together called a 'softmax'

In [ ]:
probs.shape

In [ ]:
probs[torch.arange(5), ys] # Possibility of outputs belonging to input

In [ ]:
loss =- probs[torch.arange(5), ys].log().mean()
loss

In [ ]:
# backward pass
W.grad = None # set to 0
loss.backward()

* Actually before getting started we set grads to 0 in micgrograd but in pytorc setting None more efficient
* when we apply loss.backward() all inputs effect this and has a grad value

In [ ]:
W.data += -0.1 * W.grad

* now put it all together

In [ ]:
# create the dataset
xs, ys = [], []
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# gradient descent
for k in range(10):
  
  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

* There are some differences between bigrams and neural networks:

* Bigrams do not scale well, while the strength of neural networks is their scalability.
* When you push the weights (W) towards zero in the loss function, the distribution becomes smoother and more even.
* Now, we should talk about regularization.

In [ ]:
# Training cycles, using the entire dataset -> 200 Epochs
for k in range(200):
    
    # Forward pass
    xenc = F.one_hot(xs, num_classes=27).float() # one-hot encode the names
    logits = xenc @ W # logits, different word for log-counts
    counts = logits.exp() # 'fake counts', kinda like in  the N matrix of bigram
    probs = counts / counts.sum(1, keepdims=True) # Normal distribution probabilities (this is y_pred)
    loss = -probs[torch.arange(len(probs)), ys].log().mean() + 0.01 * (W**2).mean() # regularization loss
    print(f'Loss @ iteration {k+1}: {loss}')
    
    # Backward pass
    W.grad = None # Make sure all gradients are reset
    loss.backward() # Torch kept track of what this variable is, kinda cool
    
    # Weight update
    W.data += -50 * W.grad

In [ ]:
# Finally, sample from this neural network model
# (This structure is copied from the bigram approach)
g = torch.Generator().manual_seed(2147483642)

for i in range(5):
    out = []
    ix = 0
    while True:
        # ----------
        # BEFORE:
        #p = P[ix] # Bigram explicit probability approach
        # ----------
        # NOW:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W # predict log-counts
        counts = logits.exp() # counts, equivalent to N
        p = counts / counts.sum(1, keepdims=True) # probabilities for next character
        # ----------
    
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        
        if ix == 0:
            break
    print(''.join(out))